# Tutorial 02: Inspect Your Images (Quality Control)

**Before you start:** Run this notebook from the repo (e.g. open it under `Notebooks/tutorials/`). Replace every `path/to/...` and adjust the path template `.../(condition)/(filename).*` to match your folder layout.

**Why prepare and inspect?** Microscopes sometimes save images in different sizes or formats, and occasionally masks don't perfectly align with their images. This step acts as a safety check to spot irregularities before running the heavy AI analysis, saving you time and preventing confusing errors later on.

**Note:** You can use the E. coli toy dataset from [Tutorial 03](03_core_phenotyping_workflow.ipynb) to test these steps out-of-the-box.

See [Best Practices: Data Inspection](../../docs/guides/best-practices.md#data-inspecting) and [Getting Started: Image Preprocessing](../../docs/getting-started.md#image-preprocessing).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
from phenome import PhenoMe, get_metadata_from_path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

## Step 1: Find Files and Inspect Data

A standard workflow looks like this: First, we locate your images using `find_files()`. Then, we run `inspect_data()` to check them for any errors. Finally, once everything looks good, we process them using the AI model.

In [ ]:
# Find files (with optional mask_dir for property presets)
image_dir = "path/to/images"
mask_dir = "path/to/masks"  # Optional; omit if no masks

metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
file_df = pheno.find_files(
    image_dir,
    mask_dir=mask_dir,
    metadata_fn=metadata_fn,
    extensions=[".tif", ".tiff", ".png"],
)

print(f"Found {len(file_df)} images")

## Step 2: Run Inspection and Interpret Results

Running `inspect_data()` returns a table summarizing your images. Key things it checks:

- **Dimensions & Channels:** Matches their height, width, and number of colors.
- **Errors:** Finds corrupted files or broken paths before they crash the AI.
- **shape_match:** Checks if your image and its mask perfectly overlap. If `False`, it means they are different sizes and need fixing before analysis.

**What to look for:** Most of your images should have the same dimensions. If not, you might need to crop or pad them during processing to make them consistent.

| Finding | Typical Action (when running process_images) |
|---------|---------------------------------------------|
| Common image size | The default `resize_size=224` usually works well. |
| Very small images | Consider setting `pad_size` before resizing to avoid stretching them out. |
| Multi-channel fluorescence | Use `channel_mode='split'` so the AI handles colors correctly. |
| General photography (RGB) | Use `channel_mode='combined'` as you would with standard photos. |

In [ ]:
inspect_df = pheno.inspect_data()
print(inspect_df.head(10))

## 5. Proceed to Processing

After inspecting, run `process_images()` with appropriate parameters. See [Common Workflows](../../docs/workflows.md) for full examples.

In [ ]:
# Example: process with parameters informed by inspect
# from phenome import load_dinov2_model
# model, wrapper = load_dinov2_model(device=device)
# pheno.process_images(wrapper, resize_size=224, channel_mode="split")

## Next Steps

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [04_advanced_metadata_handling.ipynb](04_advanced_metadata_handling.ipynb): Handling messy data
- [05_exploratory_analysis_and_explainability.ipynb](05_exploratory_analysis_and_explainability.ipynb): Advanced analysis after successful QC and embedding
- [Best Practices](../../docs/guides/best-practices.md): Reproducibility and error handling